<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/wav2vec_phoneme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.0 MB/s eta 0:00:00


In [2]:
!unzip timit.zip -d ./timit

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA1.WAV  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA1.WAV.wav  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA1.WRD  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA2.PHN  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA2.TXT  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA2.WAV  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA2.WAV.wav  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SA2.WRD  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1060.PHN  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1060.TXT  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1060.WAV  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1060.WAV.wav  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1060.WRD  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1690.PHN  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1690.TXT  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1690.WAV  
  inflating: ./timit/data/TRAIN/DR6/MTXS0/SI1690.WAV.wav  
  inflating: ./timit/data/TRAIN/DR6/M

In [3]:
import torch
import torchaudio
import wandb


In [4]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Tokenizer
from omegaconf import OmegaConf
import os

In [5]:
hparams = {
    "model_name": "./model",
    "batch_size": 1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "beam_size": 10
}


In [6]:


print("Starting wandb login...")
wandb.login(key="")
hparams_omegaconf = OmegaConf.create(hparams)

wandb.init(
    project="wav2vec_phoneme",
    name="base_wav2vec_phoneme",
    config=OmegaConf.to_container(hparams_omegaconf, resolve=True)
)
print("wandb initialized.")


Starting wandb login...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dkkim2008 (dkkim2008-hankuk-university-for-foreign-studies) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb initialized.


In [9]:
!git lfs install
!git clone https://huggingface.co/excalibur12/wav2vec2-large-lv60_phoneme-timit_english_timit-4k ./model


Git LFS initialized.
Cloning into './model'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 123 (delta 43), reused 0 (delta 0), pack-reused 10 (from 1)
Receiving objects: 100% (123/123), 880.15 KiB | 10.48 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [10]:

print("Loading tokenizer and model...")
tokenizer = Wav2Vec2Tokenizer.from_pretrained(hparams["model_name"])
model = Wav2Vec2ForCTC.from_pretrained(hparams["model_name"]).to(hparams["device"])
print(f"Model loaded on {hparams['device']}.")


total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 수: {total_params:,}")


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Wav2Vec2PhonemeCTCTokenizer'. 
The class this function is called from is 'Wav2Vec2Tokenizer'.


Loading tokenizer and model...


/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/tokenization_wav2vec2.py:720: FutureWarning: The class `Wav2Vec2Tokenizer` is deprecated and will be removed in version 5 of Transformers. Please use `Wav2Vec2Processor` or `Wav2Vec2CTCTokenizer` instead.
  warnings.warn(


Model loaded on cuda.
전체 파라미터 수: 315,506,370


In [ ]:



timit_path = "./timit/data/TEST"  # TIMIT 테스트셋 경로


def load_audio(file_path):
    speech, sr = torchaudio.load(file_path)
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        speech = resampler(speech)
    return speech.squeeze()


def decode_logits(logits):
    pred_ids = torch.argmax(logits, dim=-1)
    pred_ids = pred_ids.cpu()
    transcription = tokenizer.batch_decode(pred_ids)[0]
    return transcription.replace(" ", "")


def evaluate_timit(timit_dir):
    references = []
    hypotheses = []
    file_count = 0

    print(f"Starting evaluation on: {timit_dir}")
    for root, _, files in os.walk(timit_dir):
        for file in files:
            #print(file)
            if file.lower().endswith(".wav.wav"):
                file_count += 1
                print(f"Processing file {file_count}: {file}")
                wav_path = os.path.join(root, file)
                phoneme_path = wav_path[:-8] + ".PHN"
                # print(file)
                # print(phoneme_path)
                if not os.path.isfile(phoneme_path):
                    print(f"Warning: Corresponding phoneme file not found for {file}, skipping.")
                    continue

                try:
                    speech = load_audio(wav_path).to(hparams["device"])
                    input_values = tokenizer(speech.cpu(), return_tensors="pt", padding="longest").input_values.to(hparams["device"])

                    with torch.no_grad():
                        logits = model(input_values).logits

                    pred_phonemes = decode_logits(logits)

                    with open(phoneme_path, "r") as f:
                        phoneme_labels = [line.strip().split()[-1] for line in f.readlines()]
                    ref_phonemes = "".join(phoneme_labels)

                    # print(pred_phonemes)
                    # print(ref_phonemes)

                    hypotheses.append(pred_phonemes)
                    references.append(ref_phonemes)

                    wandb.log({
                        "sample_pred": pred_phonemes,
                        "sample_ref": ref_phonemes
                    })
                except Exception as e:
                    print(f"Error processing {file}: {e}")

    print("Calculating PER (WER 근사)...")
    from jiwer import wer
    per = wer(references, hypotheses)
    wandb.log({"PER_wer_approx": per})
    print(f"Phoneme Error Rate (PER, WER 근사): {per:.3f}")


if __name__ == "__main__":
    print("Evaluation script started.")
    evaluate_timit(timit_path)
    print("Evaluation script finished.")


Evaluation script started.
Starting evaluation on: ./timit/data/TEST
Processing file 1: SA1.WAV.wav
Processing file 2: SI596.WAV.wav
Processing file 3: SA2.WAV.wav
Processing file 4: SI1915.WAV.wav
Processing file 5: SI1285.WAV.wav
Processing file 6: SX295.WAV.wav
Processing file 7: SX205.WAV.wav
Processing file 8: SX25.WAV.wav
Processing file 9: SX115.WAV.wav
Processing file 10: SX385.WAV.wav
Processing file 11: SA1.WAV.wav
Processing file 12: SA2.WAV.wav
Processing file 13: SX216.WAV.wav
Processing file 14: SI2286.WAV.wav
Processing file 15: SI2118.WAV.wav
Processing file 16: SX36.WAV.wav
Processing file 17: SX396.WAV.wav
Processing file 18: SX126.WAV.wav
Processing file 19: SI1656.WAV.wav
Processing file 20: SX306.WAV.wav
Processing file 21: SA1.WAV.wav
Processing file 22: SI2168.WAV.wav
Processing file 23: SA2.WAV.wav
Processing file 24: SX368.WAV.wav
Processing file 25: SX278.WAV.wav
Processing file 26: SI908.WAV.wav
Processing file 27: SX8.WAV.wav
Processing file 28: SI1538.WAV.w